# Stage 1 — CREsted topic model evaluation

This notebook follows the CREsted topic-classification evaluation logic: load topic BED-derived AnnData, register genome, load trained model, predict on held-out regions, evaluate test-set performance, and export per-topic summaries.

Inputs expected on Alvis:
- `runs/out/human_topics.h5ad`
- `runs/out/macaque_topics.h5ad`
- `runs/out/deeptopic_human/final_model.keras`
- `runs/out/deeptopic_macaque/final_model.keras`
- pycisTopic topic annotation + QC tables in `data/`

Outputs:
- `runs/out/stage1_crested_eval/<species>/...`


In [2]:
from __future__ import annotations

import os
from pathlib import Path

os.environ.setdefault("KERAS_BACKEND", "torch")

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["pdf.fonttype"] = 42
matplotlib.rcParams["ps.fonttype"] = 42

import matplotlib.pyplot as plt

import anndata as ad
import crested
import keras

try:
    from sklearn.metrics import average_precision_score, roc_auc_score, confusion_matrix
except Exception as e:
    average_precision_score = None
    roc_auc_score = None
    confusion_matrix = None
    print("[WARN] sklearn metrics unavailable:", e)


In [3]:
# =====================
# Config
# =====================
BASE = Path("/mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt")
DATA = BASE / "data"
OUT = BASE / "runs" / "out"

OUTDIR = OUT / "stage1_crested_eval"
OUTDIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 4
MODEL_LAYER = "model_prediction"

# Keep chromosome split consistent with the CREsted topic-classification tutorial.
# If your h5ad already has var["split"], it will be reused.
DEFAULT_VAL_CHROMS = ["chr8", "chr10"]
DEFAULT_TEST_CHROMS = ["chr9", "chr18"]

DATASETS = {
    "human": {
        "adata": OUT / "human_topics.h5ad",
        "model": OUT / "deeptopic_human" / "final_model.keras",
        "genome_fasta": Path("/mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/genomes/hg38/hg38.fa"),
        "chrom_sizes": Path("/mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/genomes/hg38/hg38.chrom.sizes"),
        "annotation_candidates": DATA / "3_topic_annotation_human_pycistopic.tsv",
        "qc_candidates": DATA / "3_topic_qc_metrics_human.tsv",
    },
    "macaque": {
        "adata": OUT / "macaque_topics.h5ad",
        "model": OUT / "deeptopic_macaque" / "final_model.keras",
        "genome_fasta": Path("/mimer/NOBACKUP/groups/naiss2025-22-612/yiquan/macaque/rheMac10.fa"),
        "chrom_sizes": Path("/mimer/NOBACKUP/groups/naiss2025-22-612/yiquan/macaque/rheMac10.chrom.sizes"),
        "annotation_candidates": DATA / "3_topic_annotation_macaque_pycistopic.tsv",
        "qc_candidates": DATA / "3_topic_qc_metrics_macaque.tsv",
    },
}

RUN_SPECIES = ["human", "macaque"]


In [9]:
def first_existing(candidates):
    if candidates is None:
        return None

    # allow single Path / string
    if isinstance(candidates, (str, Path)):
        candidates = [candidates]

    for p in candidates:
        if p is not None and Path(p).exists():
            return Path(p)

    return None


def dense_array(x):
    if hasattr(x, "toarray"):
        return x.toarray()
    return np.asarray(x)


def normalize_topic_name(x):
    s = str(x)
    if s.lower().startswith("topic"):
        return "Topic" + s.replace("Topic", "").replace("topic", "")
    try:
        return f"Topic{int(float(s))}"
    except Exception:
        return s


def load_topic_annotation(candidates):
    path = first_existing(candidates)
    if path is None:
        print("[WARN] no topic annotation table found")
        return None

    print("[load annotation]", path)
    df = pd.read_csv(path, sep=None, engine="python")
    df.columns = [str(c).strip() for c in df.columns]

    if "topic" not in df.columns:
        # common pycisTopic export: first unnamed column or index-like column
        first_col = df.columns[0]
        df = df.rename(columns={first_col: "topic"})

    df["topic"] = df["topic"].map(normalize_topic_name)
    return df


def load_topic_qc(candidates):
    path = first_existing(candidates)
    if path is None:
        print("[WARN] no topic QC table found")
        return None

    print("[load topic QC]", path)
    df = pd.read_csv(path, sep=None, engine="python")
    df.columns = [str(c).strip() for c in df.columns]

    if "topic" not in df.columns:
        first_col = df.columns[0]
        df = df.rename(columns={first_col: "topic"})

    df["topic"] = df["topic"].map(normalize_topic_name)
    return df


def register_species_genome(cfg):
    chrom_sizes = cfg.get("chrom_sizes")
    if chrom_sizes is not None and Path(chrom_sizes).exists():
        genome = crested.Genome(str(cfg["genome_fasta"]), str(chrom_sizes))
    else:
        genome = crested.Genome(str(cfg["genome_fasta"]))
    crested.register_genome(genome)
    return genome


def ensure_split(adata):
    if "split" in adata.var.columns:
        print("[split] reuse existing adata.var['split']")
        print(adata.var["split"].value_counts())
        return

    print("[split] creating chr-based split")
    crested.pp.train_val_test_split(
        adata,
        strategy="chr",
        val_chroms=DEFAULT_VAL_CHROMS,
        test_chroms=DEFAULT_TEST_CHROMS,
    )
    print(adata.var["split"].value_counts())


def add_predictions_to_layer(adata, model, layer_name=MODEL_LAYER):
    print("[predict] running crested.tl.predict")
    pred = crested.tl.predict(adata, model, batch_size=4)
    pred = np.asarray(pred)

    # Tutorial stores predictions as classes × regions in adata.layers
    if pred.shape == (adata.n_vars, adata.n_obs):
        layer = pred.T
    elif pred.shape == (adata.n_obs, adata.n_vars):
        layer = pred
    else:
        raise ValueError(
            f"Unexpected prediction shape {pred.shape}; "
            f"expected {(adata.n_vars, adata.n_obs)} or {(adata.n_obs, adata.n_vars)}"
        )

    adata.layers[layer_name] = layer
    print(f"[layer] adata.layers['{layer_name}'] =", adata.layers[layer_name].shape)
    return layer


def merge_annotation_into_obs(adata, annot_df, qc_df):
    adata.obs["topic"] = [normalize_topic_name(x) for x in adata.obs_names]

    if annot_df is not None:
        annot = annot_df.copy().set_index("topic")
        for col in annot.columns:
            adata.obs[col] = adata.obs["topic"].map(annot[col])

    if qc_df is not None:
        qc = qc_df.copy().set_index("topic")
        for col in qc.columns:
            adata.obs[f"qc__{col}"] = adata.obs["topic"].map(qc[col])

    return adata


def compute_per_topic_metrics(adata, split="test", layer_name=MODEL_LAYER):
    mask = np.asarray(adata.var["split"] == split)
    y_true = dense_array(adata.X[:, mask]).T
    y_pred = dense_array(adata.layers[layer_name][:, mask]).T
    topic_names = list(adata.obs_names)

    rows = []
    for i, topic in enumerate(topic_names):
        yt = y_true[:, i]
        yp = y_pred[:, i]

        row = {
            "topic": normalize_topic_name(topic),
            "n_regions": int(len(yt)),
            "n_positive": int(np.sum(yt > 0)),
            "positive_fraction": float(np.mean(yt > 0)),
            "mean_true": float(np.mean(yt)),
            "mean_pred": float(np.mean(yp)),
        }

        if average_precision_score is not None and len(np.unique(yt > 0)) > 1:
            row["auPR"] = float(average_precision_score((yt > 0).astype(int), yp))
        else:
            row["auPR"] = np.nan

        if roc_auc_score is not None and len(np.unique(yt > 0)) > 1:
            row["auROC"] = float(roc_auc_score((yt > 0).astype(int), yp))
        else:
            row["auROC"] = np.nan

        rows.append(row)

    metrics = pd.DataFrame(rows)

    true_argmax = np.argmax(y_true, axis=1)
    pred_argmax = np.argmax(y_pred, axis=1)
    cat_acc = float(np.mean(true_argmax == pred_argmax))

    overall = pd.DataFrame([{
        "split": split,
        "n_regions": int(y_true.shape[0]),
        "n_topics": int(y_true.shape[1]),
        "categorical_accuracy_argmax": cat_acc,
        "mean_auPR": float(metrics["auPR"].mean(skipna=True)),
        "median_auPR": float(metrics["auPR"].median(skipna=True)),
        "mean_auROC": float(metrics["auROC"].mean(skipna=True)),
        "median_auROC": float(metrics["auROC"].median(skipna=True)),
    }])

    return metrics, overall, true_argmax, pred_argmax


def make_confusion_table(adata, true_argmax, pred_argmax):
    labels = list(adata.obs_names)
    if confusion_matrix is None:
        return None
    cm = confusion_matrix(true_argmax, pred_argmax, labels=np.arange(len(labels)))
    return pd.DataFrame(cm, index=labels, columns=labels)


def plot_per_topic_bars(metrics, out_png, value_col="auPR"):
    df = metrics.sort_values(value_col, ascending=False).copy()
    plt.figure(figsize=(22, 5))
    plt.bar(df["topic"], df[value_col])
    plt.xticks(rotation=90, fontsize=5)
    plt.ylabel(value_col)
    plt.xlabel("Topic")
    plt.title(f"Per-topic {value_col}")
    plt.tight_layout()
    plt.savefig(out_png, dpi=220, bbox_inches="tight")
    plt.close()
    print("[save fig]", out_png)


def plot_confusion(cm_df, out_png, max_topics=100):
    if cm_df is None:
        return
    # row normalize
    cm = cm_df.astype(float)
    cm_norm = cm.div(cm.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)

    plt.figure(figsize=(18, 16))
    im = plt.imshow(cm_norm.values, aspect="auto", vmin=0, vmax=np.nanpercentile(cm_norm.values, 99), cmap="viridis")
    plt.colorbar(im, fraction=0.03, pad=0.01, label="row-normalized count")
    plt.xticks(np.arange(cm_norm.shape[1]), cm_norm.columns, rotation=90, fontsize=5)
    plt.yticks(np.arange(cm_norm.shape[0]), cm_norm.index, fontsize=5)
    plt.xlabel("Predicted topic")
    plt.ylabel("True topic")
    plt.title("Argmax topic confusion on test regions")
    plt.tight_layout()
    plt.savefig(out_png, dpi=220, bbox_inches="tight")
    plt.close()
    print("[save fig]", out_png)


In [10]:
def run_one_species(tag, cfg):
    print(f"\n===== {tag} =====")
    species_out = OUTDIR / tag
    species_out.mkdir(parents=True, exist_ok=True)

    print("[load adata]", cfg["adata"])
    adata = ad.read_h5ad(cfg["adata"])
    print(adata)

    annot_df = load_topic_annotation(cfg["annotation_candidates"])
    qc_df = load_topic_qc(cfg["qc_candidates"])

    merge_annotation_into_obs(adata, annot_df, qc_df)

    print("[register genome]")
    genome = register_species_genome(cfg)

    ensure_split(adata)

    print("[load model]", cfg["model"])
    model = crested.utils.load_model(str(cfg["model"]))
    print(model)

    add_predictions_to_layer(adata, model, MODEL_LAYER)

    # CREsted tutorial-style evaluation function.
    # For topic classification, use CREsted's default classification config.
    print("[crested evaluate] test set")
    try:
        eval_result = crested.tl.evaluate(
            adata,
            model=MODEL_LAYER,
            metrics=crested.tl.default_configs("topic_classification"),
        )
        print(eval_result)
        pd.DataFrame(eval_result if isinstance(eval_result, list) else [eval_result]).to_csv(
            species_out / f"{tag}_crested_evaluate.tsv",
            sep="\t",
            index=False,
        )
    except Exception as e:
        print("[WARN] crested.tl.evaluate failed; custom sklearn metrics will still be exported.")
        print(type(e).__name__, e)

    metrics, overall, true_argmax, pred_argmax = compute_per_topic_metrics(adata, split="test", layer_name=MODEL_LAYER)

    # Attach pycisTopic annotation/QC to exported metrics
    if annot_df is not None:
        metrics = metrics.merge(annot_df, on="topic", how="left")
    if qc_df is not None:
        metrics = metrics.merge(qc_df, on="topic", how="left", suffixes=("", "_qc"))

    metrics.to_csv(species_out / f"{tag}_per_topic_test_metrics.tsv", sep="\t", index=False)
    overall.to_csv(species_out / f"{tag}_overall_test_metrics.tsv", sep="\t", index=False)

    cm_df = make_confusion_table(adata, true_argmax, pred_argmax)
    if cm_df is not None:
        cm_df.to_csv(species_out / f"{tag}_argmax_confusion_counts.tsv", sep="\t")

    plot_per_topic_bars(metrics, species_out / f"{tag}_per_topic_auPR.png", "auPR")
    plot_per_topic_bars(metrics, species_out / f"{tag}_per_topic_auROC.png", "auROC")
    plot_confusion(cm_df, species_out / f"{tag}_argmax_confusion_heatmap.png")

    # CREsted tutorial-style plots: prediction vs ground truth + correlation heatmap
    # These use adata.layers[MODEL_LAYER].
    try:
        top_topic = metrics.sort_values("auPR", ascending=False)["topic"].iloc[0]
        print("[plot scatter] top topic:", top_topic)
        crested.pl.corr.scatter(
            adata,
            class_name=top_topic,
            model_names=MODEL_LAYER,
            split="test",
            log_transform=False,
            square=True,
            width=8,
            height=8,
        )
        plt.savefig(species_out / f"{tag}_scatter_{top_topic}.png", dpi=220, bbox_inches="tight")
        plt.close()
    except Exception as e:
        print("[WARN] crested.pl.corr.scatter failed:", type(e).__name__, e)

    try:
        crested.pl.corr.heatmap(
            adata,
            split="test",
            log_transform=False,
            vmax=1,
            vmin=0,
        )
        fig = plt.gcf()
        fig.set_size_inches(18, 16)

        ax = plt.gca()
        ax.tick_params(axis="x", labelsize=4, rotation=90)
        ax.tick_params(axis="y", labelsize=4)

        plt.savefig(
            species_out / f"{tag}_crested_corr_heatmap.png",
            dpi=260,
            bbox_inches="tight"
        )
        plt.close()
    except Exception as e:
        print("[WARN] crested.pl.corr.heatmap failed:", type(e).__name__, e)

    # Save evaluated adata with predictions and annotations.
    out_h5ad = species_out / f"{tag}_topics_with_predictions_stage1.h5ad"
    adata.write_h5ad(out_h5ad)
    print("[save]", out_h5ad)

    return adata, metrics, overall


results = {}
for tag in RUN_SPECIES:
    results[tag] = run_one_species(tag, DATASETS[tag])



===== human =====
[load adata] /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/runs/out/human_topics.h5ad
AnnData object with n_obs × n_vars = 100 × 415405
    obs: 'file_path', 'n_open_regions'
    var: 'n_classes', 'chr', 'start', 'end', 'split'
[load annotation] /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/data/3_topic_annotation_human_pycistopic.tsv
[load topic QC] /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/data/3_topic_qc_metrics_human.tsv
[register genome]
2026-04-28T14:54:28.102332+0200 INFO Genome hg38 registered.
[split] reuse existing adata.var['split']
split
train    334833
val       41820
test      38752
Name: count, dtype: int64
[load model] /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/runs/out/deeptopic_human/final_model.keras
<Functional name=functional, built=True>
[predict] running crested.tl.predict
103852/103852 ━━━━━━━━━━━━━━━━━━━━ 2151s 21ms/step
[layer] adata.layers['model_prediction'] = (100, 415405)
[crested evaluate] test set
[WARN] crested.

/mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/miniconda3/envs/work/lib/python3.11/site-packages/crested/pl/corr/_scatter.py:244: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pearson_corr, _ = pearsonr(x, y)
/mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/miniconda3/envs/work/lib/python3.11/site-packages/crested/pl/corr/_scatter.py:245: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  spearman_corr, _ = spearmanr(x, y)


[save] /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/runs/out/stage1_crested_eval/human/human_topics_with_predictions_stage1.h5ad

===== macaque =====
[load adata] /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/runs/out/macaque_topics.h5ad
AnnData object with n_obs × n_vars = 100 × 292050
    obs: 'file_path', 'n_open_regions'
    var: 'n_classes', 'chr', 'start', 'end', 'split'
[load annotation] /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/data/3_topic_annotation_macaque_pycistopic.tsv
[load topic QC] /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/data/3_topic_qc_metrics_macaque.tsv
[register genome]
2026-04-28T15:31:08.877628+0200 INFO Genome rheMac10 registered.
[split] reuse existing adata.var['split']
split
train    232255
test      30155
val       29640
Name: count, dtype: int64
[load model] /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/runs/out/deeptopic_macaque/final_model.keras
<Functional name=functional, built=True>
[predict] running crested.tl.predict
730

/mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/miniconda3/envs/work/lib/python3.11/site-packages/crested/pl/corr/_scatter.py:244: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pearson_corr, _ = pearsonr(x, y)
/mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/miniconda3/envs/work/lib/python3.11/site-packages/crested/pl/corr/_scatter.py:245: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  spearman_corr, _ = spearmanr(x, y)


[save] /mimer/NOBACKUP/groups/naiss2025-22-612/xinyu/pjt/runs/out/stage1_crested_eval/macaque/macaque_topics_with_predictions_stage1.h5ad


## Outputs to use later

For each species, the key files are:

- `<species>_topics_with_predictions_stage1.h5ad`
- `<species>_per_topic_test_metrics.tsv`
- `<species>_overall_test_metrics.tsv`
- `<species>_argmax_confusion_counts.tsv`

Use the h5ad file in later notebooks if you want to avoid re-running prediction.
